# Classical Baselines: Logistic Regression and XGBoost for HIGGS Classification

**Project:** Higgs Event Lab | **Module:** MID-03

This notebook benchmarks Logistic Regression and XGBoost on the original UCI HIGGS dataset using three feature groups. It defaults to the documented 250,000-row analysis sample; set `HIGGS_ANALYSIS_ROWS=11000000` before setup for a full-row run if your RAM/runtime budget supports it.

> **Scientific boundary:** all events are Monte Carlo simulations. These are benchmark classification scores, not physical discovery claims.

## Step 0 — Project setup and complete-source data preparation

The cell sequence below locates/clones the repository, installs only missing/outdated dependencies, and downloads the original UCI HIGGS archive with retries and resume support. The raw file is validated before any analysis starts.

The EDA and baseline notebooks default to **250,000 analysis rows** (the documented mid-semester sample) to reduce RAM/time. Set `HIGGS_ANALYSIS_ROWS=11000000` before running the setup cell for a full-row analysis. The separate full-Colab notebook is the dedicated full-scale model run.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
from packaging.requirements import Requirement
from packaging.version import Version

REPO_URL = "https://github.com/CosmicMagnetar/higgs-event-lab.git"
cwd = Path.cwd().resolve()


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
candidates = []
if os.environ.get("HIGGS_REPO_ROOT"):
    candidates.append(Path(os.environ["HIGGS_REPO_ROOT"]).expanduser().resolve())
candidates.extend([cwd, cwd / "higgs-event-lab", *cwd.parents])

def is_higgs_repo(path):
    return (path / "src" / "higgs_event_lab").is_dir() and (path / "pyproject.toml").is_file()


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
root = next((path for path in dict.fromkeys(candidates) if is_higgs_repo(path)), None)
if root is None:
    repo_target = cwd / "higgs-event-lab"
    suffix = 1
    while repo_target.exists() and not is_higgs_repo(repo_target):
        if repo_target.is_dir() and not any(repo_target.iterdir()):
            break
        suffix += 1
        repo_target = cwd / f"higgs-event-lab-{suffix}"
    print("Cloning Higgs Event Lab into:", repo_target)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_target)], check=True)
    root = repo_target.resolve()

if not is_higgs_repo(root):
    raise FileNotFoundError(f"Could not locate a valid Higgs Event Lab checkout at {root}.")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
src_dir = root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
os.environ["HIGGS_REPO_ROOT"] = str(root)
os.chdir(root)
print("Project root:", root)


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
REQUIRED_PACKAGES = [
    ("requests", "requests>=2.31"),
    ("numpy", "numpy>=1.26"),
    ("pandas", "pandas>=2.1"),
    ("sklearn", "scikit-learn>=1.4"),
    ("matplotlib", "matplotlib>=3.8"),
]
missing_or_old = []
for module_name, requirement_text in REQUIRED_PACKAGES:
    requirement = Requirement(requirement_text)
    try:
        installed = Version(version(requirement.name))
    except PackageNotFoundError:
        installed = None
    if installed is None or (requirement.specifier and installed not in requirement.specifier):
        missing_or_old.append(requirement_text)
if missing_or_old:
    print("Installing required packages:", ", ".join(missing_or_old))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_or_old], check=True)
else:
    print("Required packages already satisfy the minimum versions.")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
raw_path = root / "data" / "raw" / "HIGGS.csv.gz"
zip_path = raw_path.parent / "higgs_download.zip"
temp_raw_path = raw_path.with_name(raw_path.name + ".part")
validation_marker = raw_path.with_name(raw_path.name + ".validation.json")
raw_path.parent.mkdir(parents=True, exist_ok=True)
output_dir = root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
url = "https://archive.ics.uci.edu/static/public/280/higgs.zip"
full_row_count = 11_000_000
analysis_row_count = int(os.environ.get("HIGGS_ANALYSIS_ROWS", "250000"))
if not 10 <= analysis_row_count <= full_row_count:
    raise ValueError(f"HIGGS_ANALYSIS_ROWS must be between 10 and {full_row_count:,}.")
processed_path = raw_path
print("Raw dataset path:", raw_path)
print(f"Analysis rows: {analysis_row_count:,}")


## Step 0B — Resumable download helpers

This cell only defines the ZIP/GZIP validation and retry/resume functions. It does not start a download.

This section records the purpose, expected output and interpretation of the computation so the result can be checked independently.


In [ ]:
import re
import time
import zipfile
from pathlib import Path


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def find_csv_gz_member(archive_path):
    """Return the CSV.GZ member only when the ZIP directory is readable."""
    archive_path = Path(archive_path)
    if not archive_path.is_file():
        return None
    try:
        with zipfile.ZipFile(archive_path, "r") as archive:
            return next((name for name in archive.namelist() if name.lower().endswith(".csv.gz")), None)
    except (OSError, zipfile.BadZipFile, EOFError):
        return None


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _download_mode(response, existing, destination):
    if response.status_code == 416:
        Path(destination).unlink(missing_ok=True)
        raise IOError("The server rejected the resume offset; restarting from zero.")
    response.raise_for_status()
    append = existing > 0 and response.status_code == 206
    if append:
        content_range = response.headers.get("Content-Range", "")
        match = re.match(r"bytes\s+(\d+)-", content_range)
        if not match or int(match.group(1)) != existing:
            Path(destination).unlink(missing_ok=True)
            raise IOError("Server returned an unexpected byte range; restarting from zero.")
    return "ab" if append else "wb"


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _download_attempt(download_url, destination, session):
    destination = Path(destination)
    existing = destination.stat().st_size if destination.exists() else 0
    headers = {"Range": f"bytes={existing}-"} if existing else {}
    with session.get(download_url, headers=headers, stream=True, timeout=(30, 180)) as response:
        mode = _download_mode(response, existing, destination)
        expected = response.headers.get("Content-Length")
        received = 0
        with destination.open(mode) as target:
            for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):
                if chunk:
                    target.write(chunk)
                    received += len(chunk)
        if expected is not None and received != int(expected):
            raise IOError(f"Incomplete HTTP response: received {received:,} of {int(expected):,} bytes.")
    if not find_csv_gz_member(destination):
        raise IOError("The downloaded ZIP is not complete yet.")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def download_zip_resumable(download_url, destination, max_attempts=10):
    """Download with retries and HTTP Range resume where the server supports it."""
    import requests
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if find_csv_gz_member(destination):
        print("A readable ZIP is already present; reusing it.")
        return destination
    session = requests.Session()
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            _download_attempt(download_url, destination, session)
            print(f"ZIP received: {destination.stat().st_size / (1024**3):.2f} GiB.")
            return destination
        except Exception as exc:
            last_error = exc
            print(f"Download attempt {attempt}/{max_attempts} failed: {exc}")
            if attempt < max_attempts:
                time.sleep(min(2 ** (attempt - 1), 30))
    raise RuntimeError(f"Could not obtain a readable UCI ZIP. Rerun this cell. Last error: {last_error}")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def count_gzip_rows(path, chunk_size=4 * 1024 * 1024):
    """Fully read a gzip file and count CSV rows without loading it into memory."""
    import gzip
    newline_count = 0
    last_byte = b""
    with gzip.open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            newline_count += chunk.count(b"\n")
            last_byte = chunk[-1:]
    if last_byte and last_byte != b"\n":
        newline_count += 1
    return newline_count


## Step 0C — Download, extract, and validate the raw HIGGS file

This cell reuses a previously validated file, repairs old partial files, or downloads and validates the complete 11-million-row source. Rerun this cell after a network failure.

In [ ]:
import json
import shutil
import zipfile


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _write_validation_marker(path, rows):
    marker = Path(path).with_name(Path(path).name + ".validation.json")
    marker.write_text(
        json.dumps({"rows": int(rows), "size_bytes": int(Path(path).stat().st_size)}, indent=2),
        encoding="utf-8",
    )


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _existing_raw_is_valid(path, expected_rows, marker_path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        return False
    try:
        metadata = json.loads(Path(marker_path).read_text(encoding="utf-8"))
        if metadata.get("rows") == expected_rows and metadata.get("size_bytes") == path.stat().st_size:
            with path.open("rb") as handle:
                return handle.read(2) == b"\x1f\x8b"
    except (OSError, ValueError, TypeError):
        pass
    try:
        rows = count_gzip_rows(path)
        if rows != expected_rows:
            print(f"Existing raw file is incomplete: expected {expected_rows:,}, found {rows:,} rows.")
            return False
        _write_validation_marker(path, rows)
        return True
    except Exception as exc:
        print("Existing raw file failed gzip validation:", exc)
        return False


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _extract_zip_member(zip_path, member_name, temp_path):
    temp_path.unlink(missing_ok=True)
    with zipfile.ZipFile(zip_path, "r") as archive:
        with archive.open(member_name, "r") as source, temp_path.open("wb") as target:
            shutil.copyfileobj(source, target, length=4 * 1024 * 1024)
    return count_gzip_rows(temp_path)


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _extract_and_validate_raw(url, zip_path, raw_path, temp_raw_path, marker_path, expected_rows):
    gz_name = find_csv_gz_member(zip_path)
    if not gz_name:
        print("No complete ZIP found; downloading or resuming the UCI archive.")
        download_zip_resumable(url, zip_path, max_attempts=10)
        gz_name = find_csv_gz_member(zip_path)
    if not gz_name:
        raise RuntimeError("No readable .csv.gz member found. Rerun the download helper cell.")
    print("Extracting dataset member:", gz_name)
    try:
        rows = _extract_zip_member(zip_path, gz_name, temp_raw_path)
        if rows != expected_rows:
            raise ValueError(f"Expected {expected_rows:,} rows, found {rows:,}.")
        temp_raw_path.replace(raw_path)
        _write_validation_marker(raw_path, rows)
    except Exception as exc:
        temp_raw_path.unlink(missing_ok=True)
        zip_path.unlink(missing_ok=True)
        raise RuntimeError("Extraction/validation failed. Incomplete files were removed; rerun this cell.") from exc
    zip_path.unlink(missing_ok=True)
    return rows


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
complete_row_count = None
if raw_path.exists() and _existing_raw_is_valid(raw_path, full_row_count, validation_marker):
    complete_row_count = full_row_count
    print("A previously validated complete HIGGS file is present; skipping download.")
else:
    raw_path.unlink(missing_ok=True)
    validation_marker.unlink(missing_ok=True)
    complete_row_count = _extract_and_validate_raw(
        url, zip_path, raw_path, temp_raw_path, validation_marker, full_row_count
    )
if complete_row_count != full_row_count:
    raise ValueError(f"Dataset has {complete_row_count:,} rows; expected {full_row_count:,}.")
print("Raw HIGGS data ready:", raw_path)
print(f"Validated row count: {complete_row_count:,}")
print(f"Compressed size: {raw_path.stat().st_size / (1024**3):.2f} GiB")


## Step 1 — Environment and Package Verification

This cell prints package versions and confirms `xgboost` is available. XGBoost is required for the tree ensemble baseline. If it is missing, install it with `pip install xgboost`.

In [ ]:
import importlib
import platform

packages = ["numpy", "pandas", "sklearn", "xgboost", "matplotlib"]
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:15s} {mod.__version__}")
    except ModuleNotFoundError:
        print(f"{pkg:15s} NOT INSTALLED — run: pip install {pkg}")

print(f"Python: {platform.python_version()}, Platform: {platform.system()}")


## Step 2 — Hyperparameter Configuration

All hyperparameters are declared here so the notebook is fully reproducible. These settings match the documented mid-semester benchmark. The training seed is fixed at 42 to ensure consistent stratified splits across all feature groups and models.

**Logistic Regression:** Uses L-BFGS with `C=1.0` (inverse regularisation). A `StandardScaler` is embedded inside an `sklearn.Pipeline` to prevent data leakage — scaling statistics are computed only on training data.

**XGBoost:** Uses `max_depth=6`, `n_estimators=150`, and `learning_rate=0.1` for a fast but competitive mid-semester baseline. A stronger configuration (depth=7, 600 trees) is used in `notebooks/03_full/`.

In [ ]:
from higgs_event_lab.classical import LogisticRegressionConfig, XGBoostConfig

FEATURE_GROUPS = ("low_level", "high_level", "all")
N_ROWS = analysis_row_count
DATA_PATH = raw_path

LR_CONFIG = LogisticRegressionConfig(
    C=1.0,
    max_iter=1000,
    solver="lbfgs",
    seed=42,
    target_signal_efficiency=0.50,
)


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
XGB_CONFIG = XGBoostConfig(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    seed=42,
    target_signal_efficiency=0.50,
    device="auto",
)

print("Analysis rows:", f"{N_ROWS:,}")
print("Logistic Regression config:", LR_CONFIG)
print("XGBoost config:", XGB_CONFIG)


## Step 3 — Complete-data loading and split verification

We load the raw HIGGS file using the project's validated chunked loader and create the canonical `higgs-split-v1` stratified partition. The exact full-data split is recorded in each output artifact.


In [ ]:
import pandas as pd
from higgs_event_lab.protocol import SplitConfig, create_stratified_splits

label_frame = pd.read_csv(
    DATA_PATH,
    header=None,
    usecols=[0],
    names=["class_label"],
    nrows=N_ROWS,
    dtype={"class_label": "float32"},
)
y_float = label_frame["class_label"].to_numpy(dtype="float32")
if not set(pd.unique(y_float)).issubset({0.0, 1.0}) or len(pd.unique(y_float)) != 2:
    raise ValueError("HIGGS labels must contain both classes 0.0 and 1.0.")
y = y_float.astype("int64")
split = create_stratified_splits(y, SplitConfig(seed=42))
print(f"Rows selected for each model run: {len(y):,}")
print("Feature groups:", {"low_level": 21, "high_level": 7, "all": 28})
print(f"Signal fraction: {y.mean():.4f}")
print(f"Split sizes — Train: {len(split.train):,}  Val: {len(split.validation):,}  Test: {len(split.test):,}")
for split_name, idx in [("Train", split.train), ("Val", split.validation), ("Test", split.test)]:
    print(f"  {split_name} signal fraction: {y[idx].mean():.4f}")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
del label_frame, split, y


## Step 4 — Logistic Regression Across Feature Groups

We train one Logistic Regression model per feature group. A scikit-learn `Pipeline` wraps `StandardScaler` and `LogisticRegression` to ensure the scaler is fit only on training data — preventing information leakage into validation and test partitions.

The operating threshold is selected on the validation partition at a target signal efficiency of 50%, then applied frozen to the held-out test set. This mirrors the practice of selecting operating points before seeing test data in real analyses.

In [ ]:
from higgs_event_lab.classical import run_feature_group_benchmarks

print("Training Logistic Regression across feature groups...")
lr_results = run_feature_group_benchmarks(
    data_path=DATA_PATH,
    row_count=N_ROWS,
    output_base_dir=output_dir,
    config=LR_CONFIG,
    model_type="logreg",
    groups=FEATURE_GROUPS,
)

print("\nLogistic Regression Results:")
print(f"{'Run Key':<25} {'Val AUC':>9} {'Test AUC':>9} {'Brier':>7} {'Time(s)':>8}")
print("-" * 62)
for key, res in lr_results.items():
    val_auc = res["validation"]["metrics"]["roc_auc"]
    test_auc = res["test"]["metrics"]["roc_auc"]
    brier = res["test"]["metrics"]["brier_score"]
    t = res["training_seconds"]
    print(f"{key:<25} {val_auc:>9.4f} {test_auc:>9.4f} {brier:>7.4f} {t:>8.2f}s")


## Step 5 — XGBoost Across Feature Groups

XGBoost uses gradient-boosted decision trees which naturally handle feature interactions and do not require normalisation. The same stratified split as Logistic Regression is used to ensure a fair comparison — only the model and its scale of operation change.

Validation data is passed as an evaluation set to enable early stopping on the XGBoost side (though here we run the full `n_estimators` for a reproducible midsem baseline). The operating threshold is again selected on the validation partition.

In [ ]:
print("Training XGBoost across feature groups...")
xgb_results = run_feature_group_benchmarks(
    data_path=DATA_PATH,
    row_count=N_ROWS,
    output_base_dir=output_dir,
    xgb_config=XGB_CONFIG,
    model_type="xgboost",
    groups=FEATURE_GROUPS,
)

print("\nXGBoost Results:")
print(f"{'Run Key':<28} {'Val AUC':>9} {'Test AUC':>9} {'Brier':>7} {'Time(s)':>8}")
print("-" * 65)
for key, res in xgb_results.items():
    val_auc = res["validation"]["metrics"]["roc_auc"]
    test_auc = res["test"]["metrics"]["roc_auc"]
    brier = res["test"]["metrics"]["brier_score"]
    t = res["training_seconds"]
    print(f"{key:<28} {val_auc:>9.4f} {test_auc:>9.4f} {brier:>7.4f} {t:>8.2f}s")


## Step 6 — ROC Curve Comparison

We plot the ROC curves for all six runs (2 models × 3 feature groups) on a single figure. The diagonal dashed line represents a random classifier (AUC = 0.5). Higher curves indicate better discrimination. This figure directly addresses the research question: *do high-level invariant mass features add measurable discriminating power for classical models?*

## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
for ax, (title, results) in zip(axes, [("Logistic Regression", lr_results), ("XGBoost", xgb_results)]):
    ax.set_facecolor("#111820")
    ax.plot([0, 1], [0, 1], color="#3d4550", linestyle="--", linewidth=1, label="Random (AUC=0.50)")
    for key, res in results.items():
        group = key.split("_", 1)[1]
        y_test = res["test"]["labels"]
        scores = res["test"]["probabilities"]
        auc = res["test"]["metrics"]["roc_auc"]
        fpr, tpr, _ = roc_curve(y_test, scores)
        ax.plot(fpr, tpr, color=GROUP_COLORS.get(group, "#ffffff"), linewidth=2,
                label=f"{group} (AUC={auc:.4f})")
    ax.set_xlabel("False Positive Rate", color="#8b949e")
    ax.set_ylabel("True Positive Rate", color="#8b949e")
    ax.set_title(title, color="#e6edf3", fontsize=12)
    ax.tick_params(colors="#8b949e")
    ax.legend(facecolor="#1e2833", edgecolor="#3d4550", labelcolor="#c9d1d9", fontsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor("#3d4550")

fig.suptitle(f"ROC Curves: Classical Baselines — Feature Group Comparison ({N_ROWS:,} events)",
             color="#e6edf3", fontsize=13, fontweight="bold")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
fig.tight_layout()
plt.savefig(output_dir / "roc_classical_comparison.png", dpi=150, bbox_inches="tight",
            facecolor="#0a0e14")
plt.show()
print("ROC curve saved to outputs/roc_classical_comparison.png")


## Step 7 — Summary table and interpretation

This table aggregates all six analysis-sample runs for comparison and direct inclusion in the project report. It highlights the measured model and feature-group combination with the best held-out ROC-AUC.


In [ ]:
import pandas as pd

rows = []
for model_label, results in [("Logistic Regression", lr_results), ("XGBoost", xgb_results)]:
    for key, res in results.items():
        group = key.split("_", 1)[1]
        m = res["test"]["metrics"]
        op = res["operating_point"]
        rows.append({
            "Model": model_label,
            "Feature Group": group,
            "# Features": 21 if group == "low_level" else (7 if group == "high_level" else 28),
            "Test ROC-AUC": round(m["roc_auc"], 4),
            "Test PR-AUC": round(m["pr_auc"], 4),
            "Brier Score": round(m["brier_score"], 4),
            "Test Accuracy": round(m.get("accuracy", float("nan")), 4),
            "Bkg Rejection @50%": round(op["test_background_rejection"], 4),
            "Train Time (s)": round(res["training_seconds"], 2),
        })

df = pd.DataFrame(rows).sort_values("Test ROC-AUC", ascending=False)
pd.set_option("display.max_columns", None)


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
pd.set_option("display.width", 150)
print(df.to_string(index=False))

best = df.iloc[0]
print(f"\nBest classical baseline: {best['Model']} on {best['Feature Group']} features")
print(f"  Test ROC-AUC: {best['Test ROC-AUC']:.4f}")
print(f"  Background rejection @ 50% signal efficiency: {best['Bkg Rejection @50%']:.4f}")


## Step 8 — Verify exported full-data artifacts

All six analysis-sample runs have been exported to `outputs/` as `higgs-experiment-v1` artifacts. This final cell verifies that each run folder contains the required metadata and held-out predictions.


In [ ]:
expected_run_ids = [
    "logreg_low_level", "logreg_high_level", "logreg_all",
    "xgboost_low_level", "xgboost_high_level", "xgboost_all",
]

print("Artifact verification:")
all_ok = True
for run_id in expected_run_ids:
    run_dir = output_dir / run_id
    metrics_ok = (run_dir / "metrics.json").is_file()
    preds_ok = (run_dir / "test_predictions.csv").is_file()
    status = "OK" if (metrics_ok and preds_ok) else "MISSING"
    if not (metrics_ok and preds_ok):
        all_ok = False
    print(f"  {run_id:<28} metrics.json={'OK' if metrics_ok else 'MISSING':7}  "
          f"test_predictions.csv={'OK' if preds_ok else 'MISSING'}  [{status}]")

print()


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
if all_ok:
    print("All 6 classical baseline artifacts are valid. Launch the dashboard with:")
    print("  streamlit run streamlit_app.py")
else:
    print("Some artifacts are missing — re-run Steps 4 and 5.")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from pathlib import Path

colors = {"low_level": "#00e5ff", "high_level": "#ffb300", "all": "#00ff9c"}
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="#0a0e14")
for axis, (title, results) in zip(axes, (("Logistic Regression", lr_results), ("XGBoost", xgb_results))):
    axis.set_facecolor("#111820")
    axis.plot([0, 1], [0, 1], "--", color="#778899", label="Random classifier")
    for key, result in results.items():
        group = key.split("_", 1)[1]
        fpr, tpr, _ = roc_curve(result["test"]["labels"], result["test"]["probabilities"])
        auc = result["test"]["metrics"]["roc_auc"]
        axis.plot(fpr, tpr, color=colors[group], label=f"{group} (AUC={auc:.4f})")
    axis.set(title=title, xlabel="False Positive Rate", ylabel="True Positive Rate", xlim=(0, 1), ylim=(0, 1.02))
    axis.legend(facecolor="#111820", edgecolor="#303944", labelcolor="white", fontsize=8)
    axis.grid(True, color="#303944", alpha=0.5)
fig.suptitle(f"ROC Curves: Classical Baselines ({N_ROWS:,} events)", color="white")
fig.tight_layout()
output_path = Path(output_dir) / "roc_classical_comparison.png"
fig.savefig(output_path, dpi=160, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("ROC image saved to:", output_path)


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
from IPython.display import display, FileLink

display(FileLink(str(png_path)))


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.
